# Prerequisites
## Install FFmpeg 
### On windows:
- Download latest `ffmpeg-release-essentials` from here: https://www.gyan.dev/ffmpeg/builds/
- Extract the archive
- Move FFmpeg to a permanent location (C:\ffmpeg). You should get something like `C:\ffmpeg\bin\ffmpeg.exe`
- Add FFmpeg to PATH: Edit the system environment variables -> Path -> New -> `C:\ffmpeg\bin`
- Verify installation: in new cmd window type `ffmpeg -version`. It should return a version
### On Linux:
Run following command:
```
!sudo apt update && sudo apt install ffmpeg
```

In [ ]:
# Check if CUDA 12.6 is installed and available. If this command doesn’t exist, you likely don’t have NVIDIA GPU.
# In that case you'll have to fall back to CPU mode, which is slower but still works.
!nvidia-smi

In [ ]:
# If you have PyTorch without CUDA support installed, uninstall it first to avoid conflicts.
!python -m pip uninstall -y torch torchvision torchaudio

In [ ]:
# Install PyTorch with CUDA 12.6 support. 
!python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!python -m pip install faster-whisper ipywidgets

In [ ]:
from faster_whisper import WhisperModel
import torch

video_file = r"source_video/Ef11-Songs-Of-The-Old-Ages.mp4"
srt_file = r"EF11 Songs of the Old Ages/Songs of the Old Ages (english).srt"
TARGET_SEGMENT_SECONDS = 4

# Auto-select GPU or CPU
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
    compute_type = "float16"
else:
    print("Using CPU")
    device = "cpu"
    compute_type = "int8"

model = WhisperModel(
    "medium.en",
    device=device,
    compute_type=compute_type
)

segments, info = model.transcribe(
    video_file,
    beam_size=5,
    language="en",
    condition_on_previous_text=False,
    vad_filter=True,
    vad_parameters=dict(
        max_speech_duration_s=TARGET_SEGMENT_SECONDS,
    ),
    log_progress=True
)

def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)

    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

with open(srt_file, "w", encoding="utf-8") as f:
    for idx, segment in enumerate(segments, start=1):
        f.write(f"{idx}\n")
        f.write(
            f"{format_timestamp(segment.start)} --> "
            f"{format_timestamp(segment.end)}\n"
        )
        f.write(segment.text.strip() + "\n\n")

print(f"Subtitle file written to {srt_file}")